# 03 Feature Engineering

This notebook builds time-series features used by the models.

Feature groups:
- calendar features
- lag features
- rolling statistics
- price returns
- exponential moving average
- volatility estimate


## Research Paper Alignment

The engineered lag/rolling/return features are the practical approximation of temporal memory and nonlinear adjustment discussed in the paper.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "params.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

def read_json(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))


In [2]:
params["feature_engineering"]


{'input_path': 'dvc_pipeline/data/processed/iron_ore_clean.csv',
 'output_path': 'dvc_pipeline/data/features/feature_dataset.csv',
 'train_output_path': 'dvc_pipeline/data/features/train.csv',
 'test_output_path': 'dvc_pipeline/data/features/test.csv',
 'feature_columns_path': 'dvc_pipeline/data/features/feature_columns.json',
 'report_path': 'dvc_pipeline/reports/feature_report.json',
 'date_column': 'Date',
 'target_column': 'Price',
 'lags': [1, 2, 3, 5, 7, 14, 21, 30],
 'rolling_windows': [7, 14, 30]}

In [3]:
subprocess.run([sys.executable, "dvc_pipeline/src/feature_engineering.py"], check=True)


CompletedProcess(args=['c:\\aditi\\.venv\\Scripts\\python.exe', 'dvc_pipeline/src/feature_engineering.py'], returncode=0)

In [4]:
report = read_json(params["feature_engineering"]["report_path"])
feature_df = pd.read_csv(params["feature_engineering"]["output_path"])
feature_columns = json.loads(Path(params["feature_engineering"]["feature_columns_path"]).read_text(encoding="utf-8"))
report, feature_df.head(), feature_columns[:10]


({'rows_after_feature_engineering': 2234,
  'train_rows': 1982,
  'test_rows': 252,
  'feature_count': 37,
  'target_column': 'Price',
  'date_column': 'Date',
  'lags': [1, 2, 3, 5, 7, 14, 21, 30],
  'rolling_windows': [7, 14, 30]},
          Date  Price   Open   High    Low  Vol.  Change %  day_of_week  \
 0  2016-02-17  44.54  44.54  44.54  44.54   0.0      0.25            2   
 1  2016-02-18  44.89  44.89  44.89  44.89   0.0      0.79            3   
 2  2016-02-19  45.19  45.01  45.01  45.01  50.0      0.67            4   
 3  2016-02-22  45.59  45.59  45.59  45.59   0.0      0.89            0   
 4  2016-02-23  45.78  45.78  45.78  45.78   0.0      0.42            1   
 
    day_of_month  week_of_year  ...  Price_roll_min_14  Price_roll_max_14  \
 0            17             7  ...              41.13              44.43   
 1            18             7  ...              41.15              44.54   
 2            19             7  ...              41.17              44.89   
 3    